# Alzheimer's OASIS-1 — HippoDeep 特征提取

白盒诊断流程的第一阶段：不训练新的 CNN，直接用已发表、验证过的小型分割网络
从原始 MRI 里提取海马体积特征：

- **HippoDeep**（`bthyreau/hippodeep_pytorch`，纯 PyTorch，CPU/GPU 均可，几秒/例）→ 左右海马体积

全脑体积不用 SynthSeg 独立计算（SynthSeg 的预训练模型权重不在仓库里，官方 UCL SharePoint
下载链接经常打不开，GitHub issue #116 长期未解决，属于不受控的外部依赖，已放弃），
改在下一个 notebook（`alzheimer_labels_merge_colab.ipynb`）里直接用 `aseg.stats`/官方 nWBV。

输入：Google Drive 上的 OASIS-1 全量数据（disc1–11，实际文件夹名是 `OASIS_Freesurfer`），
每个受试者取 `mri/brainmask.mgz`（已颅骨剥离、已配准到 FreeSurfer 标准空间）转成 NIfTI 后喂给 HippoDeep。

输出：一张特征表 `oasis_features_hippodeep_synthseg.csv`（文件名沿用，内容只有 HippoDeep 部分），
存回 Drive，供下游 labels_merge / ANFIS 训练使用。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('nibabel', 'openpyxl', 'pandas', 'tqdm')
print('Base packages installed.')

In [ ]:
# ── HippoDeep：克隆官方仓库（纯 PyTorch，无需额外深度学习框架）───────
import os

HIPPODEEP_REPO_DIR = '/content/hippodeep_pytorch'
if not os.path.isdir(HIPPODEEP_REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                            'https://github.com/bthyreau/hippodeep_pytorch.git',
                            HIPPODEEP_REPO_DIR])
else:
    print('hippodeep_pytorch already cloned.')
print('HippoDeep ready at', HIPPODEEP_REPO_DIR)

In [ ]:
# ── 路径配置（沿用 alzheimer_dsnet_oasis.ipynb 的目录约定，
# 实测 Drive 上的实际文件夹名是 OASIS_Freesurfer，不是 OASIS1_FreeSurfer）─────
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive'
OASIS_DIR  = f'{DRIVE_ROOT}/OASIS_Freesurfer'            # disc1..disc11 都在这个目录下
XLSX_PATH  = f'{OASIS_DIR}/oasis_cross-sectional-5708aa0a98d82080.xlsx'

OUTPUT_DIR      = f'{DRIVE_ROOT}/alzheimer_anfis_outputs'
NIFTI_DIR       = f'{OUTPUT_DIR}/nifti'
HIPPODEEP_DIR   = f'{OUTPUT_DIR}/hippodeep'
FEATURES_CSV    = f'{OUTPUT_DIR}/oasis_features_hippodeep_synthseg.csv'

for d in [OUTPUT_DIR, NIFTI_DIR, HIPPODEEP_DIR]:
    os.makedirs(d, exist_ok=True)

print('OASIS_DIR  :', OASIS_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)

In [ ]:
# ── 找出所有受试者的 brainmask.mgz（跨 disc1..disc11 递归搜索）────
# 只保留 MR1（过滤 MR2/MR3 重复扫描，避免同一受试者多次出现造成数据泄漏）。
#
# 用 os.walk 而不是 Path.glob('**/...')：实测 Colab 挂载的 Google Drive（FUSE）上
# pathlib 的递归 glob 经常直接返回空结果，os.walk 更可靠。

print('正在扫描 brainmask.mgz 文件（受试者数量大时可能需要几分钟）...')
mgz_map = {}
for root, dirs, files in os.walk(OASIS_DIR):
    if 'brainmask.mgz' in files and os.path.basename(root) == 'mri':
        sid = os.path.basename(os.path.dirname(root))
        mgz_map[sid] = os.path.join(root, 'brainmask.mgz')

assert mgz_map, (f'未找到任何 brainmask.mgz！请检查 OASIS_DIR={OASIS_DIR} 下是否已有下载好的 disc 数据；'
                  f'如果确认数据在但还是找不到，先跑 drive.flush_and_unmount() + '
                  f'drive.mount(force_remount=True) 重新挂载 Drive。')

mgz_mr1 = {sid: path for sid, path in mgz_map.items() if sid.endswith('_MR1')}
print(f'共找到 {len(mgz_map)} 个 MGZ 文件，过滤后 MR1 受试者：{len(mgz_mr1)} 个')

In [ ]:
# ── mri/brainmask.mgz → NIfTI（HippoDeep 吃 .nii.gz）────
# 断点续跑：已经转换过的受试者直接跳过。

import nibabel as nib
from tqdm.notebook import tqdm

nifti_paths = {}
converted, skipped = 0, 0
for sid, mgz_path in tqdm(mgz_mr1.items(), desc='mgz -> nii.gz'):
    out_path = f'{NIFTI_DIR}/{sid}.nii.gz'
    nifti_paths[sid] = out_path
    if os.path.exists(out_path):
        skipped += 1
        continue
    img = nib.load(mgz_path)
    nib.save(nib.Nifti1Image(img.get_fdata(), img.affine, img.header), out_path)
    converted += 1

print(f'新转换 {converted} 例，已存在跳过 {skipped} 例，共 {len(nifti_paths)} 例 NIfTI')

In [ ]:
# ── 跑 HippoDeep（批量，一次性把所有受试者的路径传进去）────────────
# 官方用法：python hippodeep.py file1.nii.gz file2.nii.gz ...
# 输出：{name}_mask_L.nii.gz / {name}_mask_R.nii.gz / {name}_hippoLR_volumes.csv（每例一份，
# 实测列名是 eTIV, hippoL, hippoR，单位 mm^3）
#
# 为了不把所有受试者的中间文件都堆在 NIFTI_DIR 里，按批次跑（比如每批 50 例），
# 避免单次命令行参数过长导致失败。

hippodeep_script = f'{HIPPODEEP_REPO_DIR}/hippodeep.py'
all_sids = list(nifti_paths.keys())
BATCH_SIZE = 50

def hippo_volume_csv_path(nifti_path: str) -> str:
    stem = Path(nifti_path).name.split('.nii')[0]
    return str(Path(nifti_path).parent / f'{stem}_hippoLR_volumes.csv')

todo = [sid for sid in all_sids if not os.path.exists(hippo_volume_csv_path(nifti_paths[sid]))]
print(f'待处理 {len(todo)} / {len(all_sids)} 例（已有结果的会跳过）')

for i in tqdm(range(0, len(todo), BATCH_SIZE), desc='HippoDeep 批次'):
    batch_sids = todo[i:i + BATCH_SIZE]
    batch_paths = [nifti_paths[sid] for sid in batch_sids]
    try:
        subprocess.run([sys.executable, hippodeep_script, *batch_paths],
                        check=True, cwd=HIPPODEEP_REPO_DIR)
    except subprocess.CalledProcessError as e:
        print(f'  批次 {i} 失败: {e}. 继续下一批。')

print('HippoDeep 跑完。')

In [ ]:
# ── 汇总 HippoDeep 输出（每个受试者一份 *_hippoLR_volumes.csv）─────
# 实测列名固定是 eTIV, hippoL, hippoR，单位 mm^3（不是 mL——如果是 mL 单侧海马会比整个大脑还大，
# 解剖学上不可能，已用数值范围核实过）。

import pandas as pd

hippo_rows = []
for sid, nifti_path in nifti_paths.items():
    csv_path = hippo_volume_csv_path(nifti_path)
    if not os.path.exists(csv_path):
        continue
    row = pd.read_csv(csv_path).iloc[0].to_dict()
    hippo_rows.append({'subject_id': sid, **row})

hippo_df = pd.DataFrame(hippo_rows)
print(f'汇总了 {len(hippo_df)} 例 HippoDeep 结果')
print(hippo_df.head())

In [ ]:
# ── 保存特征表到 Drive（全脑体积不在这里算，留给 labels_merge notebook 用 aseg.stats/官方nWBV）──
features_df = hippo_df[['subject_id', 'hippoL', 'hippoR', 'eTIV']].copy()
features_df['hippo_vol_hippodeep'] = features_df['hippoL'] + features_df['hippoR']
features_df = features_df.rename(columns={'eTIV': 'hippodeep_etiv'})

features_df.to_csv(FEATURES_CSV, index=False)
print(f'保存特征表: {FEATURES_CSV}  ({len(features_df)} 例)')
print('列: subject_id, hippoL, hippoR, hippodeep_etiv, hippo_vol_hippodeep')
features_df.head()